# Customer Churn Modeling

Load the prepared customer feature table for modeling.

In [1]:
import pandas as pd
import sklearn.model_selection
import sklearn.preprocessing
import sklearn.linear_model
import sklearn.ensemble
import sklearn.metrics
import xgboost
import sklearn.inspection

In [2]:
features = pd.read_csv("../data/customer_features.csv", index_col=0)

In [3]:
display(features.shape)
display(features.head())

(11414, 12)

,frequency,monetary,recency,tenure,n_bikes,n_clothing,n_accessories,country,gender,marital_status,age,churned
customer_key,,,,,,,,,,,,
1,3,8249,59,894,3,1,4,Australia,Male,Married,41.0,1
2,2,5795,166,898,2,2,3,Australia,Male,Single,37.0,0
3,3,8114,128,906,3,0,1,Australia,Male,Married,42.0,1
4,3,8139,52,915,3,1,5,Australia,Female,Single,39.0,1
5,3,8196,61,890,3,0,3,Australia,Female,Single,33.0,1


In [4]:
y = features["churned"]
x = features.drop(columns = ["churned"])

In [5]:
display(x.shape)
display(y.shape)
display(y.mean())

(11414, 11)

(11414,)

np.float64(0.6342211319432276)

In [6]:
x["country"] = x["country"].fillna("Unknown")
x["gender"] = x["gender"].fillna("Unknown")

In [7]:
display(x["country"].value_counts())
display(x["gender"].value_counts())
display(x.isna().sum())

country
America           4198
Australia         2790
United Kingdom    1214
Canada            1046
France            1035
Germany           1028
Unknown            103
Name: count, dtype: int64

gender
Male       5735
Female     5669
Unknown      10
Name: count, dtype: int64

frequency          0
monetary           0
recency            0
tenure             0
n_bikes            0
n_clothing         0
n_accessories      0
country            0
gender             0
marital_status     0
age               17
dtype: int64

In [8]:
temp = pd.get_dummies(x, columns = ["country", "gender", "marital_status"], dtype = int)

In [9]:
temp.shape

(11414, 20)

In [10]:
temp.columns

Index(['frequency', 'monetary', 'recency', 'tenure', 'n_bikes', 'n_clothing',
       'n_accessories', 'age', 'country_America', 'country_Australia',
       'country_Canada', 'country_France', 'country_Germany',
       'country_United Kingdom', 'country_Unknown', 'gender_Female',
       'gender_Male', 'gender_Unknown', 'marital_status_Married',
       'marital_status_Single'],
      dtype='str')

In [11]:
x = temp

In [12]:
X_train, X_test, Y_train, Y_test = sklearn.model_selection.train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

In [13]:
display(X_test.shape)
display(X_train.shape)
display(Y_test.shape)
display(Y_train.shape)
display(Y_test.mean())
display(Y_train.mean())


(2283, 20)

(9131, 20)

(2283,)

(9131,)

np.float64(0.6342531756460797)

np.float64(0.6342131201401818)

In [14]:
X_train.shape

(9131, 20)

In [15]:
x.head()

,frequency,monetary,recency,tenure,n_bikes,n_clothing,n_accessories,age,country_America,country_Australia,country_Canada,country_France,country_Germany,country_United Kingdom,country_Unknown,gender_Female,gender_Male,gender_Unknown,marital_status_Married,marital_status_Single
customer_key,,,,,,,,,,,,,,,,,,,,
1,3,8249,59,894,3,1,4,41.0,0,1,0,0,0,0,0,0,1,0,1,0
2,2,5795,166,898,2,2,3,37.0,0,1,0,0,0,0,0,0,1,0,0,1
3,3,8114,128,906,3,0,1,42.0,0,1,0,0,0,0,0,0,1,0,1,0
4,3,8139,52,915,3,1,5,39.0,0,1,0,0,0,0,0,1,0,0,0,1
5,3,8196,61,890,3,0,3,33.0,0,1,0,0,0,0,0,1,0,0,0,1


In [16]:
median_agge = X_train["age"].median()

In [17]:
print(median_agge)

41.0


In [18]:
X_train["age"] = X_train["age"].fillna(median_agge)
X_test["age"] = X_test["age"].fillna(median_agge)

In [19]:
##X_train.isna().sum().sum()
X_test.isna().sum().sum()

np.int64(0)

In [20]:
scalar = sklearn.preprocessing.StandardScaler()

In [21]:
X_train_scaled = scalar.fit_transform(X_train)

In [22]:
X_test_scaled = scalar.transform(X_test)

In [23]:
print(X_train_scaled)

[[-0.41869054  0.38362633 -0.79301997 ... -0.02961257  0.92453413
  -0.92453413]
 [-0.41869054 -0.24181713 -0.64354323 ... -0.02961257  0.92453413
  -0.92453413]
 [-0.41869054 -0.61243071 -0.38082654 ... -0.02961257 -1.08162584
   1.08162584]
 ...
 [-0.41869054 -0.84611288 -0.23134981 ... -0.02961257 -1.08162584
   1.08162584]
 [ 1.05002376  2.14945994 -0.74319439 ... -0.02961257  0.92453413
  -0.92453413]
 [ 1.05002376  2.28057573 -0.64354323 ... -0.02961257 -1.08162584
   1.08162584]]


In [24]:
log_reg = sklearn.linear_model.LogisticRegression(max_iter = 1500)

In [25]:
log_reg.fit(X_train_scaled, Y_train)

LogisticRegression(max_iter=1500)

In [26]:
log_reg.n_iter_

array([25], dtype=int32)

In [27]:
y_pred = log_reg.predict(X_test_scaled)

In [28]:
sklearn.metrics.accuracy_score(Y_test, y_pred)

0.8002628120893561

In [29]:
sklearn.metrics.confusion_matrix(Y_test, y_pred)

array([[ 518,  317],
       [ 139, 1309]])

In [30]:
display(sklearn.metrics.precision_score(Y_test, y_pred))
display(sklearn.metrics.recall_score(Y_test, y_pred))

0.8050430504305043

0.9040055248618785

In [31]:
y_prob = log_reg.predict_proba(X_test_scaled)[:,1]
Rvalue = sklearn.metrics.roc_auc_score(Y_test, y_prob)

In [32]:
print(Rvalue)

0.8524266384358355


In [33]:
Randomforest = sklearn.ensemble.RandomForestClassifier(n_estimators = 400, random_state = 42)
Randomforest.fit(X_train, Y_train)

RandomForestClassifier(n_estimators=400, random_state=42)

In [34]:
Y_predR = Randomforest.predict(X_test)

In [35]:
Y_predRProb = Randomforest.predict_proba(X_test)

In [36]:
accuracy = sklearn.metrics.accuracy_score(Y_test, Y_predR)
confusion = sklearn.metrics.confusion_matrix(Y_test, Y_predR)
precision = sklearn.metrics.precision_score(Y_test, Y_predR)
recall = sklearn.metrics.recall_score(Y_test, Y_predR)
roc = sklearn.metrics.roc_auc_score(Y_test, Y_predRProb[:,1])
print(accuracy)
print(confusion)
print(precision)
print(recall)
print(roc)

0.9084537888742882
[[ 693  142]
 [  67 1381]]
0.9067629678266579
0.9537292817679558
0.9612300261355741


In [37]:
Y_predtestR = Randomforest.predict(X_train)

In [38]:
acc = sklearn.metrics.accuracy_score(Y_train, Y_predtestR)

In [39]:
print(acc)

0.9998904829701019


In [40]:
rows = []

for i in [5, 10, 15]:
    newrand = sklearn.ensemble.RandomForestClassifier(n_estimators = 300, random_state = 42, min_samples_leaf = i)
    newrand.fit(X_train, Y_train)

    # train accuracy
    Y_newpred = newrand.predict(X_train)
    newacc = sklearn.metrics.accuracy_score(Y_train, Y_newpred)

    # test accuracy + AUC (same model, NO second fit)
    newrandtestpred = newrand.predict(X_test)
    newrandtestacc = sklearn.metrics.accuracy_score(Y_test, newrandtestpred)
    newrandtestprob = newrand.predict_proba(X_test)
    newrandtestROC = sklearn.metrics.roc_auc_score(Y_test, newrandtestprob[:, 1])

    d = {"min_samples_leaf": i, "Train Accuracy": newacc, "Test Accuracy": newrandtestacc, "Test AUC": newrandtestROC}
    rows.append(d)

accTable = pd.DataFrame(rows)
display(accTable)


,min_samples_leaf,Train Accuracy,Test Accuracy,Test AUC
0,5,0.931880,0.894875,0.956507
1,10,0.908334,0.885677,0.950951
2,15,0.896506,0.880859,0.945748


In [41]:
xg = xgboost.XGBClassifier(n_estimators = 300, random_state = 42)
xg.fit(X_train, Y_train)
xgpredtest = xg.predict(X_test)
xgpredtrain = xg.predict(X_train)
xgacc = sklearn.metrics.accuracy_score(Y_test, xgpredtest)
print(xgacc)
xgconf = sklearn.metrics.confusion_matrix(Y_test, xgpredtest)
print(xgconf)
xgprec = sklearn.metrics.precision_score(Y_test, xgpredtest)
print(xgprec)
xgrec = sklearn.metrics.recall_score(Y_test, xgpredtest)
print(xgrec)
xgroc = sklearn.metrics.roc_auc_score(Y_test, xg.predict_proba(X_test)[:, 1])
print(xgroc)
xgtrainacc = sklearn.metrics.accuracy_score(Y_train, xgpredtrain)
print(xgtrainacc)
xgtrainconf = sklearn.metrics.confusion_matrix(Y_train, xgpredtrain)
print(xgtrainconf)
xgtrainprec = sklearn.metrics.precision_score(Y_train, xgpredtrain)
print(xgtrainprec)
xgtrainrec = sklearn.metrics.recall_score(Y_train, xgpredtrain)
print(xgtrainrec)
xgtrainroc = sklearn.metrics.roc_auc_score(Y_train, xg.predict_proba(X_train)[:, 1])
print(xgtrainroc)

0.9075777485764345
[[ 714  121]
 [  90 1358]]
0.9181879648411089
0.9378453038674033
0.9675819631455321
0.9980286934618333
[[3325   15]
 [   3 5788]]
0.9974151301051181
0.9994819547573821
0.9999819304578548


In [42]:
print(xg.feature_importances_)

[0.04857574 0.03808709 0.09030613 0.02034718 0.11259612 0.01892327
 0.02424036 0.01238182 0.11587577 0.06176813 0.09894745 0.05923201
 0.06345328 0.0357893  0.07157083 0.00906701 0.00833431 0.1002079
 0.01029633 0.        ]


In [43]:
feature_table = pd.DataFrame(index = None,data = {"feature_name": X_train.columns,"Importance" :xg.feature_importances_})
feature_table.sort_values(by = "Importance", ascending = False)

,feature_name,Importance
8,country_America,0.115876
4,n_bikes,0.112596
17,gender_Unknown,0.100208
10,country_Canada,0.098947
2,recency,0.090306
14,country_Unknown,0.071571
12,country_Germany,0.063453
9,country_Australia,0.061768
11,country_France,0.059232
0,frequency,0.048576


In [44]:
perm = sklearn.inspection.permutation_importance(xg, X_test, Y_test, scoring="roc_auc", n_repeats = 10, random_state = 42)

In [45]:
perm_table = pd.DataFrame(index = None,data = {"feature_name": X_train.columns,"Importance" : perm.importances_mean})
perm_table.sort_values(by = "Importance", ascending = False)

,feature_name,Importance
2,recency,0.293327
1,monetary,0.085464
8,country_America,0.046038
3,tenure,0.023229
10,country_Canada,0.019818
4,n_bikes,0.013022
9,country_Australia,0.011101
11,country_France,0.008168
13,country_United Kingdom,0.007342
12,country_Germany,0.006692


In [46]:
features.groupby("country")["churned"].mean().sort_values(ascending = False)

country
America           0.771081
France            0.697585
Germany           0.650778
United Kingdom    0.574959
Canada            0.551625
Australia         0.441935
Name: churned, dtype: float64

In [47]:
churn_predictions = pd.DataFrame(index = X_test.index, data = {"predicted_churn": xg.predict(X_test), "actual_churn": Y_test, "churn_probability": xg.predict_proba(X_test)[:, 1]})

In [48]:
display(churn_predictions.head())
display(churn_predictions.shape)

,predicted_churn,actual_churn,churn_probability
customer_key,,,
15027,1,1,0.987077
9713,1,1,0.932992
4577,0,0,0.002126
4909,0,0,0.001110
13894,1,1,0.701839


(2283, 3)

In [49]:
churn_predictions["risk_level"] = pd.cut(churn_predictions["churn_probability"], bins = [0, 0.4, 0.7, 1], labels = ["Low", "Medium", "High"], include_lowest = True)

In [50]:
display(churn_predictions.head())

,predicted_churn,actual_churn,churn_probability,risk_level
customer_key,,,,
15027,1,1,0.987077,High
9713,1,1,0.932992,High
4577,0,0,0.002126,Low
4909,0,0,0.001110,Low
13894,1,1,0.701839,High


In [51]:
display(churn_predictions["risk_level"].value_counts())

risk_level
High      1408
Low        772
Medium     103
Name: count, dtype: int64

In [52]:
churn_predictions.to_csv("../data/churn_predictions.csv")